# MLP warm start vs NR cold start — 4 benchmarks

> **Note:** `nr_ik` switched to the Modern Robotics library (`mr.IKinSpace` formulation) to fix numerical errors observed with the custom implementation.

In [ ]:
import sys, json
from pathlib import Path

REPO_ROOT = Path(__file__).resolve().parent.parent if "__file__" in dir() else Path().resolve().parent
sys.path.insert(0, str(REPO_ROOT))

cfg = json.loads((REPO_ROOT / "config.json").read_text())

SAVE_DATA  = str(REPO_ROOT / cfg["SAVE_DATA"])
SAVE_MODEL = str(REPO_ROOT / cfg["SAVE_MODEL"])
NR_EOMG    = cfg["NR_EOMG"]
NR_EV      = cfg["NR_EV"]
NR_MAXITER = cfg["NR_MAXITER"]
N_TESTS    = cfg["N_TESTS"]

print(f"SAVE_DATA : {SAVE_DATA}")
print(f"SAVE_MODEL: {SAVE_MODEL}")

In [ ]:
!pip install modern-robotics -q

import numpy as np
import torch
import matplotlib.pyplot as plt
import modern_robotics as mr

from ik.kinematics.fk import FK
from ik.viperx300 import JOINT_LIMITS, SLIST as Slist, M_HOME as M_home
from ik.data.dataset import IKDataset
from ik.model.mlp import MLP

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device: {DEVICE}")

In [ ]:
train_ds = IKDataset("train", SAVE_DATA)
MinMax_X = train_ds.MinMax_X
MinMax_Y = train_ds.MinMax_Y

model = MLP(input_dim=15).to(DEVICE)
model.load_state_dict(torch.load(SAVE_MODEL, map_location=DEVICE))
model.eval()
print("model loaded")

def nr_ik(T_target, q_init):
    thetalist = np.array(q_init, dtype=float).copy()
    i = 0
    Tsb = mr.FKinSpace(M_home, Slist, thetalist)
    Vs  = mr.Adjoint(Tsb) @ mr.se3ToVec(mr.MatrixLog6(mr.TransInv(Tsb) @ T_target))
    err = np.linalg.norm(Vs[:3]) > NR_EOMG or np.linalg.norm(Vs[3:]) > NR_EV
    while err and i < NR_MAXITER:
        thetalist += np.linalg.pinv(mr.JacobianSpace(Slist, thetalist)) @ Vs
        i += 1
        Tsb = mr.FKinSpace(M_home, Slist, thetalist)
        Vs  = mr.Adjoint(Tsb) @ mr.se3ToVec(mr.MatrixLog6(mr.TransInv(Tsb) @ T_target))
        err = np.linalg.norm(Vs[:3]) > NR_EOMG or np.linalg.norm(Vs[3:]) > NR_EV
    return thetalist, not err, i

def mlp_predict(q_current, T_target):
    R6     = T_target[:2, :3].flatten().astype(np.float32)
    P      = T_target[:3, 3].astype(np.float32)
    x      = np.concatenate([R6, P, q_current.astype(np.float32)])
    x_norm = (x - MinMax_X[0]) / (MinMax_X[1] - MinMax_X[0])
    x_norm = (x_norm * 2) - 1
    with torch.no_grad():
        x_t = torch.tensor(x_norm, dtype=torch.float32).unsqueeze(0).to(DEVICE)
        y_t = model(x_t).squeeze(0).cpu().numpy()
    return (y_t + 1) / 2 * (MinMax_Y[1] - MinMax_Y[0]) + MinMax_Y[0]

def summary(label, iters, conv, pos_err):
    print(f"\n{'─'*45}")
    print(f"  {label}")
    print(f"{'─'*45}")
    print(f"  mean iterations : {np.mean(iters):>8.2f}")
    print(f"  converged       : {100*np.mean(conv):>7.1f}%")
    print(f"  mean pos err (m): {np.mean(pos_err):>8.4f}")
    print(f"  max  pos err (m): {np.max(pos_err):>8.4f}")

## Test 1 — Cold start from zeros vs MLP warm start

Most realistic scenario when no prior solution exists (robot power-on, fault reset).
Both solvers target the same random poses; cold starts from `q=[0,0,0,0,0,0]`.

In [ ]:
np.random.seed(0)
q_targets = np.random.uniform(JOINT_LIMITS[:,0], JOINT_LIMITS[:,1], size=(N_TESTS, 6))
T_targets = [FK(q) for q in q_targets]
q_zero    = np.zeros(6)

c_iters, c_conv, c_err = [], [], []
w_iters, w_conv, w_err = [], [], []

for q_true, T in zip(q_targets, T_targets):
    q_c, conv_c, it_c = nr_ik(T, q_zero)
    q_w, conv_w, it_w = nr_ik(T, mlp_predict(q_zero, T))

    c_iters.append(it_c); c_conv.append(conv_c)
    c_err.append(np.linalg.norm(T[:3,3] - FK(q_c)[:3,3]))
    w_iters.append(it_w); w_conv.append(conv_w)
    w_err.append(np.linalg.norm(T[:3,3] - FK(q_w)[:3,3]))

summary("NR cold (zeros)", c_iters, c_conv, c_err)
summary("NR warm (MLP)",   w_iters, w_conv, w_err)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(c_iters, bins=30, alpha=0.6, label="cold", color="tomato")
axes[0].hist(w_iters, bins=30, alpha=0.6, label="warm", color="steelblue")
axes[0].set_xlabel("Iterations"); axes[0].set_title("Test 1 — iterations distribution")
axes[0].legend()
axes[1].scatter(range(N_TESTS), c_iters, s=3, alpha=0.4, color="tomato", label="cold")
axes[1].scatter(range(N_TESTS), w_iters, s=3, alpha=0.4, color="steelblue", label="warm")
axes[1].set_xlabel("Sample"); axes[1].set_ylabel("Iterations")
axes[1].set_title("Test 1 — per-sample iterations"); axes[1].legend()
plt.tight_layout(); plt.show()

## Test 2 — Pick-and-place: large jumps through home position

Simulates a pick-and-place cycle: A → home → B, where home = zeros.
At each intermediate stop the robot has moved far from the next target.
Cold start uses the previous solution (q_prev); warm start uses MLP prediction.
This is where q_prev is a *bad* initialiser because the next pose is far away.

In [ ]:
np.random.seed(1)
# Sample N_TESTS/2 pick configs and N_TESTS/2 place configs — far apart
q_pick  = np.random.uniform(JOINT_LIMITS[:, 0], JOINT_LIMITS[:, 1], size=(N_TESTS // 2, 6))
q_place = np.random.uniform(JOINT_LIMITS[:, 0], JOINT_LIMITS[:, 1], size=(N_TESTS // 2, 6))

T_pick  = [FK(q) for q in q_pick]
T_place = [FK(q) for q in q_place]
T_home  = FK(np.zeros(6))

# Sequence of waypoints per cycle: pick → home → place
# At each step cold-start uses the previous q; warm-start uses MLP
c2_iters, c2_conv, c2_err = [], [], []
w2_iters, w2_conv, w2_err = [], [], []

for i in range(N_TESTS // 2):
    cycle = [
        (T_pick[i],  q_pick[i]),   # pick (true q for reference)
        (T_home,     np.zeros(6)),  # home
        (T_place[i], q_place[i]),   # place
    ]
    q_cold = np.zeros(6)  # start from home for both
    q_warm = np.zeros(6)

    for T_target, _ in cycle:
        qc, cc, ic = nr_ik(T_target, q_cold)
        qw, cw, iw = nr_ik(T_target, mlp_predict(q_warm, T_target))

        c2_iters.append(ic); c2_conv.append(cc)
        c2_err.append(np.linalg.norm(T_target[:3, 3] - FK(qc)[:3, 3]))
        w2_iters.append(iw); w2_conv.append(cw)
        w2_err.append(np.linalg.norm(T_target[:3, 3] - FK(qw)[:3, 3]))

        q_cold = qc  # cold: next init = last solution
        q_warm = qw  # warm: same, but the MLP seed was used

summary("NR cold (q_prev)", c2_iters, c2_conv, c2_err)
summary("NR warm (MLP)",    w2_iters, w2_conv, w2_err)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(c2_iters, bins=30, alpha=0.6, label="cold (q_prev)", color="tomato")
axes[0].hist(w2_iters, bins=30, alpha=0.6, label="warm (MLP)",    color="steelblue")
axes[0].set_xlabel("Iterations"); axes[0].set_title("Test 2 — iterations distribution")
axes[0].legend()
axes[1].plot(c2_iters, alpha=0.5, color="tomato",    label="cold", linewidth=0.7)
axes[1].plot(w2_iters, alpha=0.5, color="steelblue", label="warm", linewidth=0.7)
axes[1].set_xlabel("Waypoint"); axes[1].set_ylabel("Iterations")
axes[1].set_title("Test 2 — iterations per waypoint"); axes[1].legend()
plt.tight_layout(); plt.show()

## Test 3 — Same target, many starting configurations

Fixes one random target pose and solves it from N_TESTS different random starting configs.
Shows how robust the MLP seed is regardless of where the robot currently is.
Cold start always uses q_current (the random start); warm start replaces it with MLP prediction.

In [ ]:
np.random.seed(2)
q_fixed   = np.random.uniform(JOINT_LIMITS[:, 0], JOINT_LIMITS[:, 1])
T_fixed   = FK(q_fixed)
q_starts  = np.random.uniform(JOINT_LIMITS[:, 0], JOINT_LIMITS[:, 1], size=(N_TESTS, 6))

c3_iters, c3_conv, c3_err = [], [], []
w3_iters, w3_conv, w3_err = [], [], []

for q_start in q_starts:
    qc, cc, ic = nr_ik(T_fixed, q_start)
    qw, cw, iw = nr_ik(T_fixed, mlp_predict(q_start, T_fixed))

    c3_iters.append(ic); c3_conv.append(cc)
    c3_err.append(np.linalg.norm(T_fixed[:3, 3] - FK(qc)[:3, 3]))
    w3_iters.append(iw); w3_conv.append(cw)
    w3_err.append(np.linalg.norm(T_fixed[:3, 3] - FK(qw)[:3, 3]))

summary("NR cold (q_current)", c3_iters, c3_conv, c3_err)
summary("NR warm (MLP)",       w3_iters, w3_conv, w3_err)

# Also show the distance from each q_start to q_fixed vs iterations
dists = [np.linalg.norm(np.rad2deg(qs - q_fixed)) for qs in q_starts]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].scatter(dists, c3_iters, s=6, alpha=0.4, color="tomato",    label="cold")
axes[0].scatter(dists, w3_iters, s=6, alpha=0.4, color="steelblue", label="warm")
axes[0].set_xlabel("||q_start - q_target|| (deg)"); axes[0].set_ylabel("Iterations")
axes[0].set_title("Test 3 — iterations vs starting distance"); axes[0].legend()
axes[1].hist(c3_iters, bins=30, alpha=0.6, label="cold", color="tomato")
axes[1].hist(w3_iters, bins=30, alpha=0.6, label="warm", color="steelblue")
axes[1].set_xlabel("Iterations"); axes[1].set_title("Test 3 — iterations distribution")
axes[1].legend()
plt.tight_layout(); plt.show()

## Test 4 — MLP-only accuracy (no NR refinement)

Measures raw MLP quality: how close is the predicted q to the true target in task space,
without any NR refinement? This isolates the MLP's contribution and shows whether it
alone is good enough for low-precision applications (e.g. visual servoing pre-positioning).

Also computes the fraction of cases where MLP alone already satisfies NR tolerances.

In [ ]:
import modern_robotics as mr
from ik.viperx300 import SLIST as Slist, M_HOME as M_home

np.random.seed(3)
q_targets4 = np.random.uniform(JOINT_LIMITS[:, 0], JOINT_LIMITS[:, 1], size=(N_TESTS, 6))
T_targets4 = [FK(q) for q in q_targets4]
q_starts4  = np.random.uniform(JOINT_LIMITS[:, 0], JOINT_LIMITS[:, 1], size=(N_TESTS, 6))

mlp_pos_err, mlp_rot_deg, mlp_converged = [], [], []

for q_start, T_target in zip(q_starts4, T_targets4):
    q_pred = mlp_predict(q_start, T_target)
    T_pred = FK(q_pred)

    pos_err = np.linalg.norm(T_target[:3, 3] - T_pred[:3, 3])

    # Geodesic rotation error: ||omega|| from SO(3) matrix log = angle in radians
    Tsb = T_pred
    Vs      = mr.Adjoint(Tsb) @ mr.se3ToVec(mr.MatrixLog6(mr.TransInv(Tsb) @ T_target))
    omg_rad = np.linalg.norm(Vs[:3])   # geodesic angle (rad)
    v_err   = np.linalg.norm(Vs[3:])

    mlp_pos_err.append(pos_err)
    mlp_rot_deg.append(np.degrees(omg_rad))
    mlp_converged.append(omg_rad < NR_EOMG and v_err < NR_EV)

NR_EOMG_DEG = np.degrees(NR_EOMG)

print(f"\n{'─'*45}")
print(f"  MLP-only (no NR)")
print(f"{'─'*45}")
print(f"  mean pos err (m)   : {np.mean(mlp_pos_err):>8.4f}")
print(f"  median pos err (m) : {np.median(mlp_pos_err):>8.4f}")
print(f"  max  pos err (m)   : {np.max(mlp_pos_err):>8.4f}")
print(f"  mean rot err (deg) : {np.mean(mlp_rot_deg):>8.2f}")
print(f"  median rot err (deg): {np.median(mlp_rot_deg):>7.2f}")
print(f"  already converged  : {100*np.mean(mlp_converged):>7.1f}%  (no NR needed)")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(mlp_pos_err, bins=40, color="mediumpurple", alpha=0.8)
axes[0].axvline(np.mean(mlp_pos_err), color="black", linestyle="--",
                label=f"mean={np.mean(mlp_pos_err):.4f} m")
axes[0].set_xlabel("Position error (m)"); axes[0].set_title("Test 4 — MLP-only position error")
axes[0].legend()
axes[1].hist(mlp_rot_deg, bins=40, color="darkorange", alpha=0.8)
axes[1].axvline(NR_EOMG_DEG, color="red", linestyle="--",
                label=f"NR tol={NR_EOMG_DEG:.2f}°")
axes[1].axvline(np.mean(mlp_rot_deg), color="black", linestyle="--",
                label=f"mean={np.mean(mlp_rot_deg):.2f}°")
axes[1].set_xlabel("Rotation error (deg)"); axes[1].set_title("Test 4 — MLP-only rotation error")
axes[1].legend()
plt.tight_layout(); plt.show()